# 05.4 - preliminary data quality checks

Read-only QA pass over the first ingestion build. **Modifies nothing.** Specified in
[`../05.4_data_quality.md`](../05.4_data_quality.md); companions explain the models
([`models_explained.md`](models_explained.md)), the metadata
([`metadata_catalog.md`](metadata_catalog.md)) and other datasets
([`other_datasets.md`](other_datasets.md)).

Access path per the plan: DuckDB aggregates over the Parquet for the big counts, plus
a full store load where object navigation is easier (the 5.3 streaming + cached
`_store.sqlite` make a full load cheap; comfortable on a 32 GB box).

Needs the `store` extra (`uv sync --extra store`) for DuckDB. Run cells top to bottom.


## Setup

Defaults to the committed **sample seed corpus** (fast dry run). Comment the
`/ "bootstrap"` line to point at the real ingestion build under `data/lexicon/` -
that is where the real findings live.


In [ ]:
import duckdb

from lang_tools.lexicon.lemma_store import LexiconStore
from lang_tools.params.lang_tools_params import get_lang_tools_params

data_fol = get_lang_tools_params().paths.data_fol
# comment out to run on the real build (data/lexicon)
# data_fol = data_fol / "bootstrap"

LEX = data_fol / "lexicon"
# concepts is a single file; lemmas/senses are partitioned per language (PARTITIONED).
SRC = {
    "concepts": LEX / "concepts.parquet",
    "lemmas": LEX / "lemmas" / "*.parquet",
    "senses": LEX / "senses" / "*.parquet",
}
con = duckdb.connect()


def t(name: str) -> str:
    """read_parquet(...) source for a table, usable inline in SQL."""
    return f"read_parquet('{SRC[name]}')"


# --- report capture -------------------------------------------------------------
# Each check is wrapped in cap(...) / cap_text(...) so the notebook both renders to
# screen AND records a captioned section; the final cell writes report.md. Re-running
# the notebook regenerates the report. REPORT_PATH is anchored to the repo (not the cwd
# or the bootstrap swap) so it always lands beside this notebook.
from datetime import datetime, timezone  # noqa: E402

REPORT_PATH = (
    get_lang_tools_params().paths.data_fol.parent
    / "scratch_space/09_concept_model/05.4_data_quality/report.md"
)
REPORT: list[tuple[str, str, str, str]] = []  # (kind, title, desc, body)


def _md_table(rel) -> str:
    cols = list(rel.columns)
    head = "| " + " | ".join(cols) + " |"
    sep = "| " + " | ".join(["---"] * len(cols)) + " |"
    rows = [
        "| " + " | ".join("" if v is None else str(v) for v in row) + " |"
        for row in rel.fetchall()
    ]
    return "\n".join([head, sep, *rows])


def cap(title: str, desc: str, rel):
    """Record a duckdb relation as a captioned markdown table; return it so the cell
    still renders the table on screen."""
    REPORT.append(("table", title, desc, _md_table(rel)))
    return rel


def cap_text(title: str, desc: str, text: str) -> None:
    """Record a free-text block (rendered in a fenced code block); also print it."""
    print(text)
    REPORT.append(("text", title, desc, text))


def write_report() -> None:
    """Write every captured check to report.md, in capture order."""
    lines = [
        "# 05.4 data quality - report",
        "",
        f"Generated: {datetime.now(timezone.utc):%Y-%m-%d %H:%M UTC} - "
        "**auto-generated by `05.4_data_quality_checks.ipynb`** (re-run to refresh).",
        f"Corpus: `{LEX}`",
        "",
        "Read-only QA pass; nothing is modified. "
        "Spec + interpretation: [`../05.4_data_quality.md`](../05.4_data_quality.md).",
        "",
    ]
    for kind, title, desc, body in REPORT:
        lines += [f"## {title}", "", desc, ""]
        lines += [body, ""] if kind == "table" else ["```", body, "```", ""]
    REPORT_PATH.write_text("\n".join(lines) + "\n", encoding="utf-8")
    print("wrote", REPORT_PATH)


LEX, sorted(p.name for p in LEX.glob("*")) if LEX.exists() else "NO CORPUS"

## 1. Volumes (the "how many" table)


In [ ]:
# Row counts per table.
cap(
    "1. Row counts",
    "Total rows per source-of-truth table.",
    con.sql(f"""
SELECT 'concepts' AS tbl, count(*) AS n FROM {t("concepts")}
UNION ALL SELECT 'lemmas', count(*) FROM {t("lemmas")}
UNION ALL SELECT 'senses',  count(*) FROM {t("senses")}
"""),
)

In [ ]:
# Lemmas per language.
cap(
    "1. Lemmas per language",
    "Lemma rows by language (OMW member forms, kaikki-enriched).",
    con.sql(
        f"SELECT language, count(*) AS n_lemmas FROM {t('lemmas')} "
        f"GROUP BY language ORDER BY n_lemmas DESC"
    ),
)

In [ ]:
# (language, part_of_speech) matrix.
cap(
    "1. Part-of-speech by language",
    "(language, POS) matrix; POS is the OMW synset POS mapped to our labels.",
    con.sql(
        f"SELECT language, part_of_speech, count(*) AS n FROM {t('lemmas')} "
        f"GROUP BY 1, 2 ORDER BY 1, n DESC"
    ),
)

In [ ]:
# Cardinality distributions: lemmas-per-concept and concepts-per-lemma.
cap(
    "1. Cardinality distributions",
    "Lemmas per concept and concepts per lemma (min/p50/p95/max/avg).",
    con.sql(f"""
WITH per_concept AS (SELECT concept_id, count(*) n FROM {t("senses")} GROUP BY concept_id),
     per_lemma   AS (SELECT lemma_id,   count(*) n FROM {t("senses")} GROUP BY lemma_id)
SELECT 'lemmas_per_concept' AS dist,
       min(n), quantile_cont(n, 0.5) AS p50, quantile_cont(n, 0.95) AS p95,
       max(n), round(avg(n), 2) AS avg FROM per_concept
UNION ALL
SELECT 'concepts_per_lemma',
       min(n), quantile_cont(n, 0.5), quantile_cont(n, 0.95), max(n), round(avg(n), 2)
FROM per_lemma
"""),
)

### Edge reconciliation (must be zero)

`Sense` is the only membership record, so it must reference real rows and every lemma
must have at least one edge. Any nonzero here is a **transform bug** (back to phase 5).


In [ ]:
cap(
    "1. Edge reconciliation (must be zero)",
    "Senses must point at real rows and every lemma must have an edge; "
    "any nonzero is a phase-5 transform bug.",
    con.sql(f"""
SELECT 'sense->missing lemma'   AS check, count(*) AS n
  FROM {t("senses")} s LEFT JOIN {t("lemmas")} l ON s.lemma_id = l.id
  WHERE l.id IS NULL
UNION ALL
SELECT 'sense->missing concept', count(*)
  FROM {t("senses")} s LEFT JOIN {t("concepts")} c ON s.concept_id = c.id
  WHERE c.id IS NULL
UNION ALL
SELECT 'lemma with no sense', count(*)
  FROM {t("lemmas")} l LEFT JOIN {t("senses")} s ON l.id = s.lemma_id
  WHERE s.lemma_id IS NULL
"""),
)

## 2. Emptiness / degenerate cardinality

`definitions` is a `map<string,string>`; `len(map_keys(...))` is the gloss count.


In [ ]:
cap(
    "2. Emptiness / degenerate cardinality",
    "Concepts with no gloss in any language, exactly one gloss, no member (orphan), "
    "or a single member.",
    con.sql(f"""
WITH members AS (SELECT concept_id, count(*) n FROM {t("senses")} GROUP BY concept_id),
     glosses AS (SELECT id, len(map_keys(definitions)) g FROM {t("concepts")})
SELECT
  (SELECT count(*) FROM glosses WHERE g = 0)                       AS concepts_no_gloss_any_lang,
  (SELECT count(*) FROM glosses WHERE g = 1)                       AS concepts_single_gloss,
  (SELECT count(*) FROM {t("concepts")} c
     LEFT JOIN members m ON c.id = m.concept_id WHERE m.n IS NULL) AS concepts_orphan_no_member,
  (SELECT count(*) FROM members WHERE n = 1)                       AS concepts_single_member
"""),
)

In [ ]:
# Single-language vs multi-language concepts (the cross-lingual win the design is for).
cap(
    "2. Single- vs multi-language concepts",
    "Concepts whose members span one vs several languages; multi-language is the "
    "cross-lingual grouping the design exists for.",
    con.sql(f"""
WITH langs AS (
  SELECT s.concept_id, count(DISTINCT l.language) nl
  FROM {t("senses")} s JOIN {t("lemmas")} l ON s.lemma_id = l.id
  GROUP BY s.concept_id
)
SELECT count(*) FILTER (WHERE nl = 1) AS single_language,
       count(*) FILTER (WHERE nl > 1) AS multi_language,
       round(100.0 * count(*) FILTER (WHERE nl > 1) / count(*), 1) AS pct_multi
FROM langs
"""),
)

In [ ]:
# Example sentences are kaikki-only and expected very sparse: how many lemmas have 0 / 1.
cap(
    "2. Example-sentence coverage",
    "Lemmas with zero / one / many example sentences (examples are kaikki-only, "
    "expected sparse).",
    con.sql(f"""
WITH e AS (SELECT len(examples) ne FROM {t("lemmas")})
SELECT count(*) FILTER (WHERE ne = 0) AS no_example,
       count(*) FILTER (WHERE ne = 1) AS single_example,
       count(*) FILTER (WHERE ne > 1) AS multi_example FROM e
"""),
)

## 3. Cross-lingual balance + enrichment yield

Gloss coverage = of the concepts that have a lemma in language L, how many actually
carry an L gloss. Note: precise **ILI-backed vs ILI-orphan** counts are not in the
Parquet (the concept id keeps only the `hash[:12]` of the grouping key, not whether it
was `ili::` or `syn::`). `single_language` above is the practical proxy; an exact count
needs re-reading OMW or persisting the ILI as a column (a phase-8 promote candidate).


In [ ]:
cap(
    "3. Gloss coverage per language",
    "Of the concepts that have a lemma in language L, how many carry an L gloss.",
    con.sql(f"""
WITH cl AS (
  SELECT DISTINCT s.concept_id, l.language
  FROM {t("senses")} s JOIN {t("lemmas")} l ON s.lemma_id = l.id
)
SELECT cl.language,
       count(*) AS concepts_touched,
       count(*) FILTER (WHERE list_contains(map_keys(c.definitions), cl.language)) AS with_gloss,
       round(100.0 * count(*) FILTER (WHERE list_contains(map_keys(c.definitions), cl.language))
             / count(*), 1) AS pct_gloss
FROM cl JOIN {t("concepts")} c ON c.id = cl.concept_id
GROUP BY cl.language ORDER BY concepts_touched DESC
"""),
)

In [ ]:
# kaikki enrichment yield per language. The model `sources` list records whether kaikki
# touched a lemma (transform appends "kaikki" to it); it is a VARCHAR[] and exists in
# every corpus - unlike the singular on-disk `source` provenance column (real build
# only, see the provenance cell below). Membership test is list_contains, not `=`.
cap(
    "3. kaikki enrichment yield per language",
    "Lemmas whose `sources` list includes kaikki (kaikki examples/gloss touched them).",
    con.sql(f"""
SELECT language,
       count(*) AS n_lemmas,
       count(*) FILTER (WHERE list_contains(sources, 'kaikki')) AS kaikki_tagged,
       round(100.0 * count(*) FILTER (WHERE list_contains(sources, 'kaikki'))
             / count(*), 1) AS pct_kaikki
FROM {t("lemmas")} GROUP BY language ORDER BY n_lemmas DESC
"""),
)

## 4. Trust / dedup signals


In [ ]:
# POS distribution + unmapped (NULL) POS count. A NULL means an OMW code outside
# _POS_LABELS (p/x/u/...) silently became null - worth knowing.
cap(
    "4. POS distribution",
    "Mapped part-of-speech counts; a `<null>` bucket would flag OMW codes outside our map.",
    con.sql(
        f"SELECT coalesce(part_of_speech, '<null>') AS pos, count(*) AS n "
        f"FROM {t('lemmas')} GROUP BY 1 ORDER BY n DESC"
    ),
)

In [ ]:
# Slug collisions (Observation 1): concept id is c__{slug}__{hash}. Top shared slugs.
cap(
    "4. Top shared concept slugs",
    "Most-collided `c__{slug}__{hash}` slugs - a legibility issue (ids stay unique via "
    "the hash); routed to phase 8.",
    con.sql(f"""
WITH s AS (SELECT split_part(id, '__', 2) AS slug FROM {t("concepts")})
SELECT slug, count(*) AS n_concepts FROM s GROUP BY slug
HAVING count(*) > 1 ORDER BY n_concepts DESC LIMIT 20
"""),
)

In [ ]:
# Slug collision totals + the generic 'concept' fallback share.
cap(
    "4. Slug collision totals",
    "Distinct slugs vs total concepts, and the generic `concept` fallback count.",
    con.sql(f"""
WITH s AS (SELECT split_part(id, '__', 2) AS slug FROM {t("concepts")})
SELECT count(*) AS concepts,
       count(DISTINCT slug) AS distinct_slugs,
       count(*) FILTER (WHERE slug = 'concept') AS generic_concept_slug
FROM s
"""),
)

### `definition == lemma` smell (the `house` defect)

A per-language definition string that, normalized, equals one of that concept's lemma
forms in that language: a gloss that is just the word, carrying no meaning. SQL uses
`lower(strip_accents(...))` as an approximation of the package `normalize`; treat the
count as a smell estimate, then eyeball the examples.


In [ ]:
cap(
    "4. definition == lemma (the `house` smell)",
    "Per-language definitions that, normalized, equal one of the concept's lemma forms - "
    "a gloss that is just the word; the core `house` defect.",
    con.sql(f"""
WITH defs AS (
  SELECT c.id AS concept_id, unnest(map_entries(c.definitions)) AS kv
  FROM {t("concepts")} c
),
hits AS (
  SELECT d.concept_id, kv.key AS lang, kv.value AS definition, l.text AS lemma
  FROM defs d
  JOIN {t("senses")} s ON s.concept_id = d.concept_id
  JOIN {t("lemmas")}  l ON l.id = s.lemma_id AND l.language = d.kv.key
  WHERE lower(strip_accents(d.kv.value)) = l.normalized
)
SELECT (SELECT count(*) FROM hits) AS definition_equals_lemma_rows,
       (SELECT count(DISTINCT concept_id) FROM hits) AS concepts_affected
"""),
)

In [ ]:
# A sample of the offending (concept, language, definition == lemma) rows.
cap(
    "4. definition == lemma samples",
    "A sample of offending (concept, language, definition, lemma) rows.",
    con.sql(f"""
WITH defs AS (
  SELECT c.id AS concept_id, unnest(map_entries(c.definitions)) AS kv
  FROM {t("concepts")} c
)
SELECT d.concept_id, d.kv.key AS lang, d.kv.value AS definition, l.text AS lemma
FROM defs d
JOIN {t("senses")} s ON s.concept_id = d.concept_id
JOIN {t("lemmas")}  l ON l.id = s.lemma_id AND l.language = d.kv.key
WHERE lower(strip_accents(d.kv.value)) = l.normalized
LIMIT 25
"""),
)

In [ ]:
# Suspicious lemmas a cleanup pass might drop/flag.
cap(
    "4. Suspicious lemmas",
    "Multi-word, digit-bearing, or very long member forms a cleanup pass might drop/flag.",
    con.sql(f"""
SELECT count(*) FILTER (WHERE text LIKE '% %')              AS multiword,
       count(*) FILTER (WHERE regexp_matches(text, '[0-9]')) AS has_digit,
       count(*) FILTER (WHERE length(text) > 30)             AS very_long
FROM {t("lemmas")}
"""),
)

**POS agreement (OMW vs kaikki)** is _not_ computable from the built tables - the
kaikki POS is dropped at ingestion (see [`metadata_catalog.md`](metadata_catalog.md)).
Computing it needs a re-parse of the raw kaikki dump (`WikiRecord.pos`) joined by
`(normalized, language)`; left as a follow-up when that check is prioritized.


## 5. Provenance & licensing snapshot (feeds phase 10)

`senses` carry no text, so by policy they are 100% `omw`; a `kaikki` sense would be a
bug. The `kaikki` share elsewhere is the CC-BY-SA-touched surface.


In [ ]:
# Provenance snapshot. The per-row `source` tag (omw|kaikki|llm|manual) is an on-disk-
# only column written by the phase-5 provenance-aware dump; the sample/seed corpus is
# built without it, so guard on its presence rather than erroring.
def _has_source(name: str) -> bool:
    cols = [r[0] for r in con.execute(f"DESCRIBE SELECT * FROM {t(name)}").fetchall()]
    return "source" in cols


parts = [
    f"SELECT '{name}' AS tbl, source, count(*) AS n FROM {t(name)} GROUP BY source"
    for name in ("lemmas", "concepts", "senses")
    if _has_source(name)
]
if parts:
    cap(
        "5. Provenance snapshot",
        "Per-row source tag counts (omw|kaikki|llm|manual); the kaikki share is the "
        "CC-BY-SA surface for phase 10.",
        con.sql(" UNION ALL ".join(parts) + " ORDER BY tbl, n DESC"),
    )
else:
    cap_text(
        "5. Provenance snapshot",
        "Per-row source tag counts (omw|kaikki|llm|manual).",
        "No `source` column in this corpus (the sample/seed build omits provenance). "
        "Run on the real phase-5 build under data/lexicon/ for the omw/kaikki tally.",
    )

## Spot-check: the `house` definition defect (via the store)

Reproduces the transform-notebook spot-check and shows, per concept `house` belongs to,
the per-language definitions next to the member forms - so the empty/`house`-only
glosses are visible directly. Uses a full store load (cheap with the 5.3 cache).


In [ ]:
store = LexiconStore.from_data_fol(data_fol)

langs = ["en", "pt", "es", "fr", "it"]
houses = [lem for lem in store.get_lemmas_by_language("en") if lem.text == "house"]
_house_lines: list[str] = []
for house in houses:
    for concept in store.concepts_for_lemma(house.id):
        _house_lines.append(f"concept: {concept.id}")
        _house_lines.append(f"  definitions: {concept.definitions}")
        for lang in langs:
            forms = [
                lem.text for lem in store.lemmas_for_concept(concept.id, language=lang)
            ]
            if forms:
                _house_lines.append(f"  {lang}: {forms}")
        _house_lines.append("")

cap_text(
    "house spot-check (the defect made visible)",
    "Every concept the English lemma `house` belongs to, with per-language definitions "
    "next to member forms. Watch the es/pt/fr definitions that are the bare word "
    "`house`/`room` or a form-of note - the sense-blind kaikki join attaching a lemma's "
    "most-common gloss to the wrong synset.",
    "\n".join(_house_lines),
)

## Output

Each check above both renders on screen and is captured (via `cap` / `cap_text`); the
final cell calls `write_report()` to write [`report.md`](report.md) beside this notebook
- so **re-running the notebook regenerates the report**. Each finding routes to its
fixing phase per [`../05.4_data_quality.md`](../05.4_data_quality.md) (8 = gloss/slug/POS
cleanup, 6 = freq/CEFR, 7 = relations, 10 = licensing). Re-run after the five-language
build and after any phase-8 cleanup to see whether quality moved - this notebook is the
lightweight regression harness.

In [ ]:
# Write report.md from everything captured above (run after all cells above).
write_report()